<a href="https://colab.research.google.com/github/Rupashi-Maurya-05/LumenDetect/blob/main/YOLOV5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connecting to drive


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

source_root = "/content/drive/MyDrive/ExDark_input"       # change if needed
mini_root   = "/content/drive/MyDrive/exdark_mini"  # output folder

images_src = os.path.join(source_root, "images")
ann_src    = os.path.join(source_root, "annotations")

images_dst = os.path.join(mini_root, "images")
ann_dst    = os.path.join(mini_root, "annotations")

os.makedirs(images_dst, exist_ok=True)
os.makedirs(ann_dst, exist_ok=True)


In [ ]:
!ls /content/drive/MyDrive


'05204092025 (1).jpg'   ExDark_input.zip	   resume+github.pdf
'05204092025 (2).jpg'   exdark_mini		   rupashi_052_dfs_file.pdf
 05204092025.jpg       'ilovepdf_merged (1).pdf'
 Classroom	       'IT workshop '


In [ ]:
import zipfile, os

zip_path = "/content/drive/MyDrive/ExDark_input.zip"   # check spelling!
extract_path = "/content/ExDark"                        # extract to Colab

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
!ls /content/ExDark


Extracted to: /content/ExDark
ExDark


In [ ]:
!find /content/ExDark -maxdepth 2 -type d


/content/ExDark
/content/ExDark/ExDark
/content/ExDark/ExDark/labels
/content/ExDark/ExDark/images


In [ ]:
images_src = "/content/ExDark/ExDark/images"
ann_src    = "/content/ExDark/ExDark/labels"


In [ ]:
mini_root   = "/content/drive/MyDrive/ExDark_mini"
images_dst  = f"{mini_root}/images"
ann_dst     = f"{mini_root}/labels"


mini dataset generation


In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm

# -------------------------
# PATHS
# -------------------------
source_dir = "/content/ExDark/ExDark"   # your extracted dataset
target_mini = "/content/ExDark_Mini"
target_clahe = "/content/ExDark_Mini_CLAHE"

# -------------------------
# CLASS MAPPING (SAME AS YOLOv5)
# -------------------------
class_mapping = {2: 0, 6: 1, 7: 2, 8: 3, 11: 4}

# -------------------------
# CLAHE SETUP (SAME AS YOLOv5)
# -------------------------
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def process_split(split):
    print(f"\n📌 Processing {split}")

    # Create required folders
    for root in [target_mini, target_clahe]:
        os.makedirs(f"{root}/images/{split}", exist_ok=True)
        os.makedirs(f"{root}/labels/{split}", exist_ok=True)

    label_dir = f"{source_dir}/labels/{split}"

    if not os.path.exists(label_dir):
        print(f"❌ Missing folder: {label_dir}")
        return

    label_files = sorted(os.listdir(label_dir))
    kept_count = 0

    for label_file in tqdm(label_files):

        if label_file.endswith(".cache"):
            continue

        src_label_path = f"{label_dir}/{label_file}"

        # Read annotation
        with open(src_label_path, "r") as f:
            lines = f.readlines()

        new_lines = []
        keep_image = False

        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])

            # keep only 5 classes
            if cls_id in class_mapping:
                new_id = class_mapping[cls_id]
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                keep_image = True

        # skip if irrelevant image
        if not keep_image:
            continue

        kept_count += 1

        # Save updated labels (baseline + CLAHE)
        for root in [target_mini, target_clahe]:
            with open(f"{root}/labels/{split}/{label_file}", "w") as f:
                f.writelines(new_lines)

        # load image
        img_name = label_file.replace(".txt", ".jpg")
        src_img_path = f"{source_dir}/images/{split}/{img_name}"

        # fallback to png
        if not os.path.exists(src_img_path):
            img_name = img_name.replace(".jpg", ".png")
            src_img_path = src_img_path.replace(".jpg", ".png")

        # read image
        img = cv2.imread(src_img_path)
        if img is None:
            continue

        # save raw
        cv2.imwrite(f"{target_mini}/images/{split}/{img_name}", img)

        # save CLAHE enhanced
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l2 = clahe.apply(l)
        lab2 = cv2.merge((l2, a, b))
        img2 = cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)

        cv2.imwrite(f"{target_clahe}/images/{split}/{img_name}", img2)

    print(f"✅ Kept {kept_count} images for {split}")

# -------------------------
# RUN BOTH SPLITS
# -------------------------
process_split("train")
process_split("val")

print("\n🎉 DATA GENERATION COMPLETE!")
print("➡ Dataset 1: ExDark_Mini")
print("➡ Dataset 2: ExDark_Mini_CLAHE")



📌 Processing train


100%|██████████| 4853/4853 [00:49<00:00, 98.84it/s]


✅ Kept 1951 images for train

📌 Processing val


100%|██████████| 1387/1387 [00:14<00:00, 93.32it/s]

✅ Kept 546 images for val

🎉 DATA GENERATION COMPLETE!
➡ Dataset 1: ExDark_Mini
➡ Dataset 2: ExDark_Mini_CLAHE


In [ ]:
import shutil

drive_target = "/content/drive/MyDrive/exdark_mini"
drive_target2 = "/content/drive/MyDrive/exdark_mini_clahe"

shutil.copytree("/content/ExDark_Mini", drive_target, dirs_exist_ok=True)
shutil.copytree("/content/ExDark_Mini_CLAHE", drive_target2, dirs_exist_ok=True)

print("✅ Copied to Google Drive!")


✅ Copied to Google Drive!


In [ ]:
!pip install torch torchvision albumentations --quiet


In [ ]:
import os
import torch
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader

class ExDarkMiniDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # load image
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        # load annotation
        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.strip().split())

                    # convert YOLO → XYXY
                    x1 = (x - bw/2) * w
                    y1 = (y - bh/2) * h
                    x2 = (x + bw/2) * w
                    y2 = (y + bh/2) * h

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls)+1)  # class IDs must start at 1

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transforms:
            img = np.array(img)
            augmented = self.transforms(image=img)
            img = augmented["image"]

        return img, target


In [ ]:
train_data = ExDarkMiniDataset(
    img_dir="/content/ExDark_Mini/images/train",
    label_dir="/content/ExDark_Mini/labels/train"
)

val_data = ExDarkMiniDataset(
    img_dir="/content/ExDark_Mini/images/val",
    label_dir="/content/ExDark_Mini/labels/val"
)

train_loader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_data, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

num_classes = 13  # ExDark has 12 classes + background

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

device = torch.device("cuda")
model.to(device)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 179MB/s]


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [ ]:
img, tgt = dataset_train[0]
print(type(img), img.shape)


NameError: name 'dataset_train' is not defined

In [ ]:
---------------------------------------------------------------------------------

In [ ]:
train_img_dir = "/content/ExDark_Mini/images/train"
train_lbl_dir = "/content/ExDark_Mini/labels/train"

val_img_dir = "/content/ExDark_Mini/images/val"
val_lbl_dir = "/content/ExDark_Mini/labels/val"

dataset_train = ExDarkMiniDataset(train_img_dir, train_lbl_dir)
dataset_val = ExDarkMiniDataset(val_img_dir, val_lbl_dir)

print("Datasets ready!")
print("Train samples:", len(dataset_train))
print("Val samples:", len(dataset_val))


Datasets ready!
Train samples: 1876
Val samples: 529


In [ ]:
img, tgt = dataset_train[0]
print(type(img), img.shape)


<class 'torch.Tensor'> torch.Size([3, 375, 500])


In [ ]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
import os
import cv2
import numpy as np


In [ ]:
class ExDarkMiniDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # Load BGR → RGB
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        img = F.to_tensor(img)   # ★ IMPORTANT ★ converts to tensor

        lbl_path = os.path.join(
            self.lbl_dir, os.path.splitext(img_name)[0] + ".txt"
        )

        boxes = []
        labels = []

        with open(lbl_path) as f:
            for line in f:
                cls, xc, yc, w, h = map(float, line.split())

                labels.append(int(cls) + 1)

                x_min = xc - w/2
                y_min = yc - h/2
                x_max = xc + w/2
                y_max = yc + h/2

                boxes.append([x_min, y_min, x_max, y_max])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        return img, target


In [ ]:
train_img_dir = "/content/ExDark_Mini/images/train"
train_lbl_dir = "/content/ExDark_Mini/labels/train"

dataset_train = ExDarkMiniDataset(train_img_dir, train_lbl_dir)


In [ ]:
img, tgt = dataset_train[0]
print(type(img), img.shape)


<class 'torch.Tensor'> torch.Size([3, 375, 500])


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(dataset_train, batch_size=4,
                          shuffle=True, collate_fn=collate_fn)


In [ ]:
imgs, tgts = next(iter(train_loader))
print(type(imgs[0]), imgs[0].shape)


<class 'torch.Tensor'> torch.Size([3, 675, 900])


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torch.optim import SGD

device = torch.device("cuda")

model = fasterrcnn_resnet50_fpn(num_classes=6)
model.to(device)

optimizer = SGD(model.parameters(), lr=0.005, momentum=0.9)

num_epochs = 3

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")
    model.train()
    total_loss = 0

    for imgs, targets in train_loader:
        imgs = [img.to(device) for img in imgs]  # ← WILL WORK NOW
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch loss:", total_loss)



--- Epoch 1/3 ---


KeyboardInterrupt: 

In [ ]:
--------------------------------------------------------------------------------------

In [ ]:
#YOLOV5

In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm

# --- CONFIGURATION ---
source_dir = "/content/yolov5/datasets/ExDark"
target_mini = "/content/yolov5/datasets/ExDark_Mini"           # Baseline Data
target_clahe = "/content/yolov5/datasets/ExDark_Mini_CLAHE"     # Enhanced Data

# Map 5 classes to IDs 0-4
# Bottle(2)->0, Chair(6)->1, Cup(7)->2, Dog(8)->3, Table(11)->4
class_mapping = {
    2: 0,   # Bottle
    6: 1,   # Chair
    7: 2,   # Cup
    8: 3,   # Dog
    11: 4   # Table
}

# CLAHE Setup (The correction technique)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def process_data(split):
    print(f"Filtering and Processing {split} data...")

    # Create directories
    for root in [target_mini, target_clahe]:
        os.makedirs(f"{root}/images/{split}", exist_ok=True)
        os.makedirs(f"{root}/labels/{split}", exist_ok=True)

    # Get file list
    if not os.path.exists(f"{source_dir}/labels/{split}"):
        print(f"Skipping {split} (folder not found)")
        return

    label_files = os.listdir(f"{source_dir}/labels/{split}")
    kept_count = 0

    for label_file in tqdm(label_files):
        src_label_path = f"{source_dir}/labels/{split}/{label_file}"

        # 1. FILTER LABELS
        with open(src_label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        keep_image = False

        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])

            # Check if object is in our Top 5
            if cls_id in class_mapping:
                new_id = class_mapping[cls_id]
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                keep_image = True

        # 2. SAVE IF VALID
        if keep_image:
            kept_count += 1
            # A. Save Labels (Identical for both)
            for root in [target_mini, target_clahe]:
                with open(f"{root}/labels/{split}/{label_file}", 'w') as f:
                    f.writelines(new_lines)

            # B. Process Image
            img_name = label_file.replace('.txt', '.jpg')
            src_img_path = f"{source_dir}/images/{split}/{img_name}"

            # Check for .png if .jpg missing
            if not os.path.exists(src_img_path):
                src_img_path = src_img_path.replace('.jpg', '.png')
                img_name = img_name.replace('.jpg', '.png')

            if os.path.exists(src_img_path):
                img = cv2.imread(src_img_path)
                if img is None: continue

                # Save RAW to Mini (Baseline)
                cv2.imwrite(f"{target_mini}/images/{split}/{img_name}", img)

                # Apply CLAHE & Save to Mini_CLAHE (Enhanced)
                lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
                l, a, b = cv2.split(lab)
                l_enhanced = clahe.apply(l)
                lab_enhanced = cv2.merge((l_enhanced, a, b))
                img_final = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)

                cv2.imwrite(f"{target_clahe}/images/{split}/{img_name}", img_final)

    print(f"  -> Kept {kept_count} images.")

# Run it
process_data('train')
process_data('val')
print("\n✅ Step 2 Complete! Datasets generated.")

Filtering and Processing train data...
Skipping train (folder not found)
Filtering and Processing val data...
Skipping val (folder not found)

✅ Step 2 Complete! Datasets generated.


In [ ]:
import os
import zipfile

# 1. Check if the zip file exists in Drive
zip_path = "/content/drive/MyDrive/ExDark_data.zip"

if os.path.exists(zip_path):
    print(f"✅ Found zip file at: {zip_path}")

    # 2. Check what is INSIDE the zip file (First 5 files)
    print("Checking zip structure...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()[:5]
        for f in file_list:
            print(f"   - {f}")

    # 3. Unzip it again (Force Overwrite)
    print("\nExtracting again...")
    !unzip -o -q "/content/drive/MyDrive/ExDark_data.zip" -d /content/yolov5/datasets/

    # 4. Check results
    contents = os.listdir("/content/yolov5/datasets/")
    print(f"\n📂 Folders currently in 'datasets': {contents}")
else:
    print("❌ Error: Zip file NOT found in Google Drive. Please check the name.")

✅ Found zip file at: /content/drive/MyDrive/ExDark_data.zip
Checking zip structure...
   - ExDark/
   - ExDark/images/
   - ExDark/images/test/
   - ExDark/images/test/2015_00021.jpg
   - ExDark/images/test/2015_00023.jpg

Extracting again...

📂 Folders currently in 'datasets': ['ExDark_Mini', 'ExDark', 'ExDark_Mini_CLAHE']


In [ ]:
import os

base_path = "/content/yolov5/datasets/ExDark"

print("🔍 Inspecting ExDark folder structure...")

if os.path.exists(base_path):
    print(f"✅ Base folder found: {base_path}")
    print(f"   Contents: {os.listdir(base_path)}")

    # Check Images
    img_path = os.path.join(base_path, "images")
    if os.path.exists(img_path):
        print(f"   📂 Images folder contents: {os.listdir(img_path)}")
    else:
        print("   ❌ 'images' folder NOT found!")

    # Check Labels
    lbl_path = os.path.join(base_path, "labels")
    if os.path.exists(lbl_path):
        print(f"   📂 Labels folder contents: {os.listdir(lbl_path)}")
    else:
        print("   ❌ 'labels' folder NOT found! (This is likely the problem)")
else:
    print("❌ Error: ExDark base folder still not found.")

🔍 Inspecting ExDark folder structure...
✅ Base folder found: /content/yolov5/datasets/ExDark
   Contents: ['labels', 'images']
   📂 Images folder contents: ['train', 'val', 'test']
   📂 Labels folder contents: ['train.cache', 'val.cache', 'train', 'val', 'test']


In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm

# --- CONFIGURATION ---
source_dir = "/content/yolov5/datasets/ExDark"
target_mini = "/content/yolov5/datasets/ExDark_Mini"
target_clahe = "/content/yolov5/datasets/ExDark_Mini_CLAHE"

# Map 5 classes: Bottle(2), Chair(6), Cup(7), Dog(8), Table(11) -> 0-4
class_mapping = {2: 0, 6: 1, 7: 2, 8: 3, 11: 4}

# CLAHE Setup
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def process_data(split):
    print(f"Processing {split} data...")

    # Create directories
    for root in [target_mini, target_clahe]:
        os.makedirs(f"{root}/images/{split}", exist_ok=True)
        os.makedirs(f"{root}/labels/{split}", exist_ok=True)

    # Path to labels
    label_dir = f"{source_dir}/labels/{split}"

    # Double check if folder exists (It should now!)
    if not os.path.exists(label_dir):
        print(f"❌ Error: {label_dir} not found.")
        return

    label_files = os.listdir(label_dir)
    kept_count = 0

    for label_file in tqdm(label_files):
        # Ignore cache files
        if label_file.endswith('.cache'): continue

        src_label_path = f"{label_dir}/{label_file}"

        # 1. Read & Filter
        with open(src_label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        keep_image = False

        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])

            if cls_id in class_mapping:
                new_id = class_mapping[cls_id]
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                keep_image = True

        # 2. Save if valid
        if keep_image:
            kept_count += 1
            # Save Labels
            for root in [target_mini, target_clahe]:
                with open(f"{root}/labels/{split}/{label_file}", 'w') as f:
                    f.writelines(new_lines)

            # Find Image
            img_name = label_file.replace('.txt', '.jpg')
            src_img_path = f"{source_dir}/images/{split}/{img_name}"

            # Check for png
            if not os.path.exists(src_img_path):
                src_img_path = src_img_path.replace('.jpg', '.png')
                img_name = img_name.replace('.jpg', '.png')

            if os.path.exists(src_img_path):
                img = cv2.imread(src_img_path)
                if img is None: continue

                # Save Raw (Baseline)
                cv2.imwrite(f"{target_mini}/images/{split}/{img_name}", img)

                # Save Enhanced (CLAHE)
                lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
                l, a, b = cv2.split(lab)
                l_enhanced = clahe.apply(l)
                lab_enhanced = cv2.merge((l_enhanced, a, b))
                img_final = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)

                cv2.imwrite(f"{target_clahe}/images/{split}/{img_name}", img_final)

    print(f"✅ Processed {split}: Kept {kept_count} images.")

# Run it
process_data('train')
process_data('val')

Processing train data...


100%|██████████| 4853/4853 [00:50<00:00, 96.17it/s]


✅ Processed train: Kept 1951 images.
Processing val data...


100%|██████████| 1387/1387 [00:14<00:00, 96.92it/s]

✅ Processed val: Kept 546 images.


In [ ]:
import os
import shutil

print("🔧 Checking environment...")

# 1. Check if the 'data' folder is missing
if not os.path.exists('/content/yolov5/data'):
    print("⚠️ YOLOv5 folder structure is incomplete.")

    # 2. Backup your processed datasets (So we don't lose them!)
    if os.path.exists('/content/yolov5/datasets'):
        print("📦 Backing up your datasets...")
        shutil.move('/content/yolov5/datasets', '/content/temp_datasets')

    # 3. Remove the broken folder
    if os.path.exists('/content/yolov5'):
        shutil.rmtree('/content/yolov5')

    # 4. Clone the code again
    print("⬇️ Downloading YOLOv5 code...")
    !git clone https://github.com/ultralytics/yolov5

    # 5. Restore your datasets
    print("📂 Restoring datasets...")
    if os.path.exists('/content/temp_datasets'):
        shutil.move('/content/temp_datasets', '/content/yolov5/datasets')

print("✅ Environment Fixed.")

# --- NOW CREATE THE CONFIG FILES ---

# 1. Config for Baseline
yaml_mini = """
path: /content/yolov5/datasets/ExDark_Mini
train: images/train
val: images/val
nc: 5
names: ['Bottle', 'Chair', 'Cup', 'Dog', 'Table']
"""
with open('/content/yolov5/data/exdark_mini.yaml', 'w') as f:
    f.write(yaml_mini)

# 2. Config for CLAHE
yaml_clahe = """
path: /content/yolov5/datasets/ExDark_Mini_CLAHE
train: images/train
val: images/val
nc: 5
names: ['Bottle', 'Chair', 'Cup', 'Dog', 'Table']
"""
with open('/content/yolov5/data/exdark_clahe.yaml', 'w') as f:
    f.write(yaml_clahe)

print("✅ Config files created successfully!")

🔧 Checking environment...
⚠️ YOLOv5 folder structure is incomplete.
📦 Backing up your datasets...
⬇️ Downloading YOLOv5 code...
Cloning into 'yolov5'...
remote: Enumerating objects: 17739, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 17739 (delta 59), reused 32 (delta 32), pack-reused 17641 (from 3)
Receiving objects: 100% (17739/17739), 17.11 MiB | 16.22 MiB/s, done.
Resolving deltas: 100% (12045/12045), done.
📂 Restoring datasets...
✅ Environment Fixed.
✅ Config files created successfully!


In [ ]:
# 1. Train Baseline (Raw Images)
!python train.py --img 640 --batch 16 --epochs 20 --data data/exdark_mini.yaml --cfg models/yolov5s.yaml --weights yolov5s.pt --name YOLOv5_Baseline --project /content/drive/MyDrive/LumenDetect

# 2. Train Enhanced (CLAHE Images)
!python train.py --img 640 --batch 16 --epochs 20 --data data/exdark_clahe.yaml --cfg models/yolov5s.yaml --weights yolov5s.pt --name YOLOv5_CLAHE --project /content/drive/MyDrive/LumenDetect

python3: can't open file '/content/train.py': [Errno 2] No such file or directory
python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [ ]:
# 1. Move inside the YOLOv5 folder
%cd /content/yolov5

# 2. NOW run the training commands
!python train.py --img 640 --batch 16 --epochs 20 --data data/exdark_mini.yaml --cfg models/yolov5s.yaml --weights yolov5s.pt --name YOLOv5_Baseline --project /content/drive/MyDrive/LumenDetect

!python train.py --img 640 --batch 16 --epochs 20 --data data/exdark_clahe.yaml --cfg models/yolov5s.yaml --weights yolov5s.pt --name YOLOv5_CLAHE --project /content/drive/MyDrive/LumenDetect

/content/yolov5
Traceback (most recent call last):
  File "/content/yolov5/train.py", line 47, in <module>
    from ultralytics.utils.patches import torch_load
ModuleNotFoundError: No module named 'ultralytics'
Traceback (most recent call last):
  File "/content/yolov5/train.py", line 47, in <module>
    from ultralytics.utils.patches import torch_load
ModuleNotFoundError: No module named 'ultralytics'


In [ ]:
# 1. Make sure we are in the folder
%cd /content/yolov5

# 2. Install the missing libraries (This fixes the 'ModuleNotFoundError')
!pip install -r requirements.txt

# 3. Start the Training (Baseline)
!python train.py --img 640 --batch 16 --epochs 20 --data data/exdark_mini.yaml --cfg models/yolov5s.yaml --weights yolov5s.pt --name YOLOv5_Baseline --project /content/drive/MyDrive/LumenDetect

# 4. Start the Training (Enhanced)
!python train.py --img 640 --batch 16 --epochs 20 --data data/exdark_clahe.yaml --cfg models/yolov5s.yaml --weights yolov5s.pt --name YOLOv5_CLAHE --project /content/drive/MyDrive/LumenDetect

Streaming output truncated to the last 5000 lines.
      19/19      5.45G   0.003627   0.004133    0.01902         50        640:  70% 83/118 [00:41<00:14,  2.46it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      19/19      5.45G   0.003632   0.004131    0.01901         48        640:  71% 84/118 [00:41<00:16,  2.10it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      19/19      5.45G   0.003632    0.00413    0.01891         53        640:  72% 85/118 [00:41<00:12,  2.55it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      19/19      5.45G   0.003631 

In [ ]:
import os

# 1. Check if YOLOv5 code exists. If not, clone it.
if not os.path.exists('/content/yolov5'):
    print("⬇️ YOLOv5 code missing. Cloning now...")
    !git clone https://github.com/ultralytics/yolov5
    %cd /content/yolov5
    !pip install -r requirements.txt
else:
    print("✅ YOLOv5 code found.")

# 2. Create the Test Config Files
print("\n📝 Creating Test Config Files...")

# Config for Baseline
yaml_mini_test = """
path: /content/yolov5/datasets/ExDark_Mini
train: images/train
val: images/test  # Validation uses Test set for final numbers
test: images/test
nc: 5
names: ['Bottle', 'Chair', 'Cup', 'Dog', 'Table']
"""
with open('/content/yolov5/data/exdark_mini_test.yaml', 'w') as f:
    f.write(yaml_mini_test)

# Config for CLAHE
yaml_clahe_test = """
path: /content/yolov5/datasets/ExDark_Mini_CLAHE
train: images/train
val: images/test  # Validation uses Test set for final numbers
test: images/test
nc: 5
names: ['Bottle', 'Chair', 'Cup', 'Dog', 'Table']
"""
with open('/content/yolov5/data/exdark_clahe_test.yaml', 'w') as f:
    f.write(yaml_clahe_test)

print("✅ Success! Config files created: exdark_mini_test.yaml & exdark_clahe_test.yaml")

⬇️ YOLOv5 code missing. Cloning now...
Cloning into 'yolov5'...
remote: Enumerating objects: 17752, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 17752 (delta 62), reused 32 (delta 32), pack-reused 17650 (from 4)
Receiving objects: 100% (17752/17752), 17.11 MiB | 23.12 MiB/s, done.
Resolving deltas: 100% (12048/12048), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.7 MB/s eta 0:00:00

📝 Creating Test Config Files...
✅ Success! Config files created: exdark_mini_test.yaml & exdark_clahe_test.yaml


In [ ]:
# 1. Config for Baseline TESTING
yaml_mini_test = """
path: /content/yolov5/datasets/ExDark_Mini
train: images/train
val: images/test  # <--- POINTING TO TEST SET
test: images/test
nc: 5
names: ['Bottle', 'Chair', 'Cup', 'Dog', 'Table']
"""
with open('/content/yolov5/data/exdark_mini_test.yaml', 'w') as f:
    f.write(yaml_mini_test)

# 2. Config for CLAHE TESTING
yaml_clahe_test = """
path: /content/yolov5/datasets/ExDark_Mini_CLAHE
train: images/train
val: images/test  # <--- POINTING TO TEST SET
test: images/test
nc: 5
names: ['Bottle', 'Chair', 'Cup', 'Dog', 'Table']
"""
with open('/content/yolov5/data/exdark_clahe_test.yaml', 'w') as f:
    f.write(yaml_clahe_test)

print("✅ Test Configs Created: exdark_mini_test.yaml & exdark_clahe_test.yaml")

✅ Test Configs Created: exdark_mini_test.yaml & exdark_clahe_test.yaml


In [ ]:
import os
import cv2
from tqdm import tqdm

# --- CONFIGURATION ---
source_dir = "/content/yolov5/datasets/ExDark"
target_mini = "/content/yolov5/datasets/ExDark_Mini"
target_clahe = "/content/yolov5/datasets/ExDark_Mini_CLAHE"

class_mapping = {2: 0, 6: 1, 7: 2, 8: 3, 11: 4} # Bottle, Chair, Cup, Dog, Table
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

print("✨ Generating TEST Data (Baseline + CLAHE)...")

def process_test_data():
    # 1. Create Test Directories
    for root in [target_mini, target_clahe]:
        os.makedirs(f"{root}/images/test", exist_ok=True)
        os.makedirs(f"{root}/labels/test", exist_ok=True)

    label_dir = f"{source_dir}/labels/test"

    # Check if source exists
    if not os.path.exists(label_dir):
        print("❌ Error: Source 'labels/test' not found. Did you unzip ExDark?")
        return

    label_files = os.listdir(label_dir)
    kept_count = 0

    for label_file in tqdm(label_files):
        if label_file.endswith(".cache"): continue

        src_label_path = f"{label_dir}/{label_file}"

        # Read & Filter
        with open(src_label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        keep_image = False

        for line in lines:
            parts = line.strip().split()
            if int(parts[0]) in class_mapping:
                new_id = class_mapping[int(parts[0])]
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                keep_image = True

        # Save if Valid
        if keep_image:
            kept_count += 1

            # A. Save Labels (Both)
            for root in [target_mini, target_clahe]:
                with open(f"{root}/labels/test/{label_file}", 'w') as f:
                    f.writelines(new_lines)

            # B. Process Image
            img_name = label_file.replace('.txt', '.jpg')
            src_img = f"{source_dir}/images/test/{img_name}"
            if not os.path.exists(src_img): src_img = src_img.replace('.jpg', '.png')

            if os.path.exists(src_img):
                img = cv2.imread(src_img)
                if img is not None:
                    # Save Raw (Baseline)
                    cv2.imwrite(f"{target_mini}/images/test/{img_name}", img)

                    # Save CLAHE (Enhanced)
                    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
                    l, a, b = cv2.split(lab)
                    l_enh = clahe.apply(l)
                    res = cv2.cvtColor(cv2.merge((l_enh, a, b)), cv2.COLOR_LAB2BGR)
                    cv2.imwrite(f"{target_clahe}/images/test/{img_name}", res)

    print(f"\n✅ Processed {kept_count} Test Images.")

process_test_data()

✨ Generating TEST Data (Baseline + CLAHE)...
❌ Error: Source 'labels/test' not found. Did you unzip ExDark?


In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm
from google.colab import drive

# 1. Connect Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Unzip Source Data (The "Repair" Step)
zip_path = "/content/drive/MyDrive/ExDark_data.zip"
source_dir = "/content/yolov5/datasets/ExDark"

if not os.path.exists(f"{source_dir}/labels/test"):
    print("📦 Unzipping source data... (Restoring missing files)")
    if not os.path.exists('/content/yolov5/datasets'):
        os.makedirs('/content/yolov5/datasets')
    # Quietly unzip
    os.system(f'unzip -o -q "{zip_path}" -d /content/yolov5/datasets/')
else:
    print("✅ Source data already exists.")

# 3. Generate Test Data (The "Processing" Step)
print("\n✨ Generating 5-Class TEST Set (Baseline + CLAHE)...")

target_mini = "/content/yolov5/datasets/ExDark_Mini"
target_clahe = "/content/yolov5/datasets/ExDark_Mini_CLAHE"
class_mapping = {2: 0, 6: 1, 7: 2, 8: 3, 11: 4} # 5 Classes
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def process_test():
    # Create Test Folders
    for root in [target_mini, target_clahe]:
        os.makedirs(f"{root}/images/test", exist_ok=True)
        os.makedirs(f"{root}/labels/test", exist_ok=True)

    label_dir = f"{source_dir}/labels/test"

    # Safety Check
    if not os.path.exists(label_dir):
        print("❌ ERROR: Still cannot find 'labels/test'. Check your Zip file structure!")
        # Debug: Print what IS there
        print(f"Contents of {source_dir}/labels: {os.listdir(f'{source_dir}/labels')}")
        return

    label_files = os.listdir(label_dir)
    kept_count = 0

    for label_file in tqdm(label_files):
        if label_file.endswith(".cache"): continue

        src_label_path = f"{label_dir}/{label_file}"

        # Read & Filter
        with open(src_label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        keep_image = False

        for line in lines:
            parts = line.strip().split()
            if int(parts[0]) in class_mapping:
                new_id = class_mapping[int(parts[0])]
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                keep_image = True

        # Save if valid
        if keep_image:
            kept_count += 1

            # A. Save Labels
            for root in [target_mini, target_clahe]:
                with open(f"{root}/labels/test/{label_file}", 'w') as f:
                    f.writelines(new_lines)

            # B. Process Image
            img_name = label_file.replace('.txt', '.jpg')
            src_img = f"{source_dir}/images/test/{img_name}"
            if not os.path.exists(src_img): src_img = src_img.replace('.jpg', '.png')

            if os.path.exists(src_img):
                img = cv2.imread(src_img)
                if img is not None:
                    # Save Baseline
                    cv2.imwrite(f"{target_mini}/images/test/{img_name}", img)

                    # Save CLAHE
                    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
                    l, a, b = cv2.split(lab)
                    l_enh = clahe.apply(l)
                    res = cv2.cvtColor(cv2.merge((l_enh, a, b)), cv2.COLOR_LAB2BGR)
                    cv2.imwrite(f"{target_clahe}/images/test/{img_name}", res)

    print(f"\n✅ Success! Processed {kept_count} Test Images.")

process_test()


Mounted at /content/drive
📦 Unzipping source data... (Restoring missing files)

✨ Generating 5-Class TEST Set (Baseline + CLAHE)...


100%|██████████| 694/694 [00:09<00:00, 70.30it/s] 


✅ Success! Processed 281 Test Images.


In [ ]:
%cd /content/yolov5

print("\n" + "="*40)
print("🚀 EVALUATING YOLOv5 BASELINE (Raw Data)")
print("="*40)
# Run validation on the TEST set
!python val.py --data data/exdark_mini_test.yaml \
               --weights /content/drive/MyDrive/LumenDetect/YOLOv5_Baseline/weights/best.pt \
               --batch 32 --img 640 --task test --name YOLOv5_Baseline_Test

print("\n" + "="*40)
print("🚀 EVALUATING YOLOv5 ENHANCED (CLAHE Data)")
print("="*40)
# Run validation on the TEST set
!python val.py --data data/exdark_clahe_test.yaml \
               --weights /content/drive/MyDrive/LumenDetect/YOLOv5_CLAHE/weights/best.pt \
               --batch 32 --img 640 --task test --name YOLOv5_CLAHE_Test

/content/yolov5

🚀 EVALUATING YOLOv5 BASELINE (Raw Data)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
val: data=data/exdark_mini_test.yaml, weights=['/content/drive/MyDrive/LumenDetect/YOLOv5_Baseline/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=YOLOv5_Baseline_Test, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-450-g781b9d57 Python-3.12.12 torch-2.9.0+cu126 CPU

Fusing layers... 
YOLOv5s summary: 157 layers, 7023610 parameters, 0 gradients, 15.8 GFLOPs
100% 755k/755k [00:00<00:00, 17.3MB/s]
test: Scanning

In [ ]:
# --- PHASE 2: Faster R-CNN Training Script (FIXED) ---
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
import cv2
import os
import glob
import numpy as np
import time

# --- CONFIG ---
DATA_ROOT = "/content/yolov5/datasets/ExDark_Mini_CLAHE"
SAVE_PATH = "/content/drive/MyDrive/LumenDetect/FasterRCNN_CLAHE.pth"
BATCH_SIZE = 8
NUM_EPOCHS = 10
NUM_CLASSES = 5

# 1. Setup Dataset (Updated to find PNGs and JPGs)
class ExDarkDataset(Dataset):
    def __init__(self, root, split):
        self.root = root
        self.split = split

        # FIXED: Look for both .jpg and .png files
        img_dir = f"{root}/images/{split}"
        self.imgs = sorted(glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png") + glob.glob(f"{img_dir}/*.jpeg"))

        self.labels_dir = f"{root}/labels/{split}"

        print(f"✅ Found {len(self.imgs)} images for {split}")

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        img = cv2.imread(img_path)
        if img is None:
             # Fallback if image is corrupt
             return self.__getitem__((idx + 1) % len(self.imgs))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        img /= 255.0

        # Load YOLO Labels
        # Note: We check for .txt matching the filename (ignoring extension)
        basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.labels_dir, basename + ".txt")

        boxes, labels = [], []
        h, w, _ = img.shape

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = list(map(float, line.strip().split()))
                    cls_id = int(parts[0]) + 1 # Add 1 (0 is background)
                    cx, cy, bw, bh = parts[1], parts[2], parts[3], parts[4]

                    xmin = (cx - bw/2) * w
                    ymin = (cy - bh/2) * h
                    xmax = (cx + bw/2) * w
                    ymax = (cy + bh/2) * h

                    # Clip boxes to image boundaries
                    xmin = max(0, xmin)
                    ymin = max(0, ymin)
                    xmax = min(w, xmax)
                    ymax = min(h, ymax)

                    # Only keep valid boxes
                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(cls_id)

        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
            target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx]), "area": area, "iscrowd": iscrowd}
        else:
            target = {"boxes": torch.zeros((0, 4), dtype=torch.float32), "labels": torch.zeros((0,), dtype=torch.int64), "image_id": torch.tensor([idx]), "area": torch.zeros((0,), dtype=torch.float32), "iscrowd": torch.zeros((0,), dtype=torch.int64)}

        img_tensor = torch.as_tensor(img.transpose((2, 0, 1)), dtype=torch.float32)
        return img_tensor, target

In [ ]:
# --- PHASE 2: Faster R-CNN Training Script (FIXED) ---
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
import cv2
import os
import glob
import numpy as np
import time

# --- CONFIG ---
DATA_ROOT = "/content/yolov5/datasets/ExDark_Mini_CLAHE"
SAVE_PATH = "/content/drive/MyDrive/LumenDetect/FasterRCNN_CLAHE.pth"
BATCH_SIZE = 8
NUM_EPOCHS = 10
NUM_CLASSES = 5

# 1. Setup Dataset (Updated to find PNGs and JPGs)
class ExDarkDataset(Dataset):
    def __init__(self, root, split):
        self.root = root
        self.split = split

        # FIXED: Look for both .jpg and .png files
        img_dir = f"{root}/images/{split}"
        self.imgs = sorted(glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png") + glob.glob(f"{img_dir}/*.jpeg"))

        self.labels_dir = f"{root}/labels/{split}"

        print(f"✅ Found {len(self.imgs)} images for {split}")

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        img = cv2.imread(img_path)
        if img is None:
             # Fallback if image is corrupt
             return self.__getitem__((idx + 1) % len(self.imgs))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        img /= 255.0

        # Load YOLO Labels
        # Note: We check for .txt matching the filename (ignoring extension)
        basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.labels_dir, basename + ".txt")

        boxes, labels = [], []
        h, w, _ = img.shape

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = list(map(float, line.strip().split()))
                    cls_id = int(parts[0]) + 1 # Add 1 (0 is background)
                    cx, cy, bw, bh = parts[1], parts[2], parts[3], parts[4]

                    xmin = (cx - bw/2) * w
                    ymin = (cy - bh/2) * h
                    xmax = (cx + bw/2) * w
                    ymax = (cy + bh/2) * h

                    # Clip boxes to image boundaries
                    xmin = max(0, xmin)
                    ymin = max(0, ymin)
                    xmax = min(w, xmax)
                    ymax = min(h, ymax)

                    # Only keep valid boxes
                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(cls_id)

        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
            target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx]), "area": area, "iscrowd": iscrowd}
        else:
            target = {"boxes": torch.zeros((0, 4), dtype=torch.float32), "labels": torch.zeros((0,), dtype=torch.int64), "image_id": torch.tensor([idx]), "area": torch.zeros((0,), dtype=torch.float32), "iscrowd": torch.zeros((0,), dtype=torch.int64)}

        img_tensor = torch.as_tensor(img.transpose((2, 0, 1)), dtype=torch.float32)
        return img_tensor, target

    def __len__(self):
        return len(self.imgs)

def collate_fn(batch):
    return tuple(zip(*batch))

# 2. Load Data
train_dataset = ExDarkDataset(DATA_ROOT, 'train')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# 3. Load Model
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)

# 4. Setup Training
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# 5. Train Loop
print(f"🚀 Starting Faster R-CNN Training on {device}...")

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    lr_scheduler.step()
    print(f"✅ Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {epoch_loss/len(train_loader):.4f} | Time: {(time.time()-start_time)/60:.1f} min")

# 6. Save Model
torch.save(model.state_dict(), SAVE_PATH)
print(f"\n🎉 Training Complete. Model saved to Drive!")

✅ Found 0 images for train


ValueError: num_samples should be a positive integer value, but got num_samples=0